<img src="./ccsf.png" alt="CCSF Logo" width=200px style="margin:0px -5px">

# Homework 13: Classification

## References

* [Classification](https://ccsf-math-108.github.io/textbook/chapters/17/classification/)
* [`datascience` Documentation](https://datascience.readthedocs.io/)
* [Code Reference](https://ccsf-math-108.github.io/materials-fa26/resources/code-reference.html)
* [Electroencephalography (EEG)](https://en.wikipedia.org/wiki/Electroencephalography)

---

## Assignment Reminders

- 🚨 Make sure to run the code cell at the top of this notebook that starts with `# Initialize Otter` to load the auto-grader.
- Your Tasks are categorized as auto-graded (📍) and manually graded (📍🔎).
    - For all the auto-graded tasks:
        - Replace the `...` in the provided code cell with your own code.
        - Run the `grader.check` code cell to run some tests on your code.
        - Keep in mind that for homework and project assignments, sometimes there are hidden tests that you will not be able to see the results of that we use for scoring the correctness of your response. **Passing the auto-grader does not guarantee that your answer is correct.**
    - For all the manually graded tasks:
        - You might need to provide your own response to the provided prompt. Do so by replacing the template text "_Type your answer here, replacing this text._" with your own words.
        - You might need to produce a graphic or something else using code. Do so by replacing the `...` in the code cell to generate the image, table, etc.
        - In either case, review the rubric on the associated <a href="https://ccsf.instructure.com" target="_blank">Canvas</a> Assignment page to understand the scoring criteria.
- Throughout this assignment and all future ones, please be sure to not re-assign variables throughout the notebook! _For example, if you use `max_temperature` in your answer to one question, do not reassign it later on. Otherwise, you will fail tests that you thought you were passing previously!_
- You may [submit](#Submit-Your-Assignment-to-Canvas) this assignment as many times as you want before the deadline. Your instructor will score the last version you submit once the deadline has passed.
- We encourage you to discuss this assignment with others but make sure to write and submit your own code. Refer to the syllabus to learn more about how to learn cooperatively.

---

---

## Set Up Notebook

### Task 00 📍

<!-- BEGIN QUESTION -->

Run the following code cells to set up the notebook.

_You are not responsible for understanding what this code does._

_Points:_ 0

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw13.ipynb")

In [ ]:
# Configure this Notebook
import numpy as np
from datascience import *
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

<!-- END QUESTION -->

---

## Introduction

### Brain signals and EEG

An **electroencephalogram (EEG)** records the electrical activity of the brain using small electrodes placed on the scalp. 

<img src="./spike-waves.png" alt="Epileptic spike and wave discharges monitored by EEG" width="200px">

Each electrode produces a **channel**: a wiggly line that shows how the voltage (measured in microvolts, µV) at that spot changes over time. The recording machine measures each channel 128 times per second, so one minute of recording from 16 channels gives $16 \times 60 \times 128 = 122{,}880$ numbers for a single person. EEG is used by researchers and doctors to study how the brain works and how brain activity differs between people.

<img src="./electrodes.jpg" alt="A drawing of a head seen from above with the EEG electrode positions marked" width="300px">

The picture above shows where the electrodes are placed. The letters tell you the region of the brain under each electrode: **F**rontal, **T**emporal, **C**entral, **P**arietal, and **O**ccipital (the back of the head). Odd numbers are on the left side, even numbers are on the right side, and **z** means the middle line. The picture also shows `Fp1`, `Fp2`, and `Fz`, which were not used in the study described below.

### The study

The recordings used in this assignment come from [a study of adolescents carried out at the Mental Health Research Center and M.V. Lomonosov Moscow State University](http://brain.bio.msu.ru/eeg_schizophrenia.htm). A psychiatrist screened every participant and placed them in one of two groups:

* **Control**: 39 healthy adolescents.
* **Schizophrenia**: 45 adolescents with symptoms of schizophrenia.

Each participant sat at rest for one minute while 16 electrodes recorded their brain activity. The question the researchers asked is whether a few numbers that summarize a person's brain waves can help tell the two groups apart.

### Features that summarize a recording

A 16-channel, one-minute recording is far too large to compare directly, so the researchers summarized each person's recording with a few numbers called **features**. The idea behind them is the **complexity** of a signal, which asks: _how hard is it to reconstruct the signal when some of its data is missing?_ Keep a fraction $S$ of the points, fill in the missing values, and measure the reconstruction error $\epsilon$. A simple, smooth signal is easy to rebuild, so its error is small. A complex signal is harder to rebuild, so its error is larger. For real signals, the researchers found that

$$\log \epsilon \approx A + B \log S,$$

so the **intercept** $A$ and the **slope** $B$ of this line summarize the complexity of the signal. The features $A_{\text{diff}}$ and $B_{\text{diff}}$ are computed the same way, but from the **changes** between consecutive measurements of the signal instead of from the signal itself.

To keep things simple, you do **not** need to compute any of these features in this assignment. They are already computed, one row per person, in the table `features` that you will load below. Your job is to use them to classify people.

### Classifying with $k$-nearest neighbors

To classify a new person, $k$-nearest neighbors (k-NN) looks at the $k$ people whose features are closest to theirs, and predicts the group that most of those neighbors belong to. "Closest" is measured with the straight-line (Euclidean) distance between the feature values. Because the features are on different scales, first convert each feature to standard units.

The researchers reported that a machine learning classifier using these four complexity features classified people into the correct group with about **89% accuracy**. They used a different method of classification. In this assignment, you will build your own k-NN classifier and measure how well it does, and you will check whether your results are consistent with that number. 

## Exploring the Data

The table `features` has one row per adolescent. The column `subject` is an identifier, `group` is either `Control` or `Schizophrenia`, and the other four columns are the features described above. Run the next two cells to load the data and confirm the group sizes.

In [ ]:
# Just run this cell!
features = Table.read_table('eeg_features.csv')
features.show(5)

In [ ]:
# Just run this cell!
features.group('group')

### Task 01 📍🔎

<!-- BEGIN QUESTION -->

Make a scatter plot of `B` (vertical axis) against `A` (horizontal axis), with a different color for each `group`.

_Points:_ 1

In [ ]:
...

# Keep this code to give your image a title
plt.title('Features A and B by Group')
plt.show()

<!-- END QUESTION -->

---

## Training, Validation, and Test Sets

To build and evaluate a classifier, you will split the 84 people into three groups:

* **Training set** (42 people): the people with known groups that the classifier uses to make predictions.
* **Validation set** (18 people): people you use to compare choices, such as the value of $k$. You use them to decide, but the classifier does not learn from them.
* **Test set** (24 people): people you use only once, at the very end, to estimate how well the final classifier works on people it has never seen.

The `split` method assigns people to the three groups at random. The test set must not influence any choice you make, which is why you should use the validation set to choose $k$.

### Task 02 📍

Use the `split` method to create `train` with 42 randomly chosen rows, `valid` with 18 of the remaining rows, and `test` with the last 24 rows. 

**Note**: `np.random.seed(2)` is set to make sure everyone gets the same data and the auto-grader will work correctly. Do not change that code.

**Hint**: `tbl.split(k)` returns two tables. The first table contains `k` randomly chosen rows, and the second contains the remaining rows. You will need to call `split` twice.

_Points:_ 4

In [ ]:
np.random.seed(2) # Do not change this line

...
print('Training set:', train.num_rows, 'people.  Validation set:', valid.num_rows, 'people.  Test set:', test.num_rows, 'people.')

In [ ]:
grader.check("task_02")

---

### Task 03 📍

Features are measured on different scales. For example, `A` takes values around $-5$, while `B_diff` takes values around $-0.3$. If you compute distances with the raw values, the feature with the largest spread would dominate. To give every feature an equal say, convert each one to **standard units**.

Compute the means and standard deviations from the **training set only**, and then use those same numbers to convert the validation and test sets. This keeps the validation and test sets unseen by the classifier.

Complete the following:

1. Assign `feature_labels` to an array with the four feature labels, `'A'`, `'B'`, `'A_diff'`, and `'B_diff'`.
2. Define `standard_units_from(values, reference)`, which converts the array `values` to standard units using the mean and standard deviation of the array `reference`.
3. Assign `train_std`, `valid_std`, and `test_std` to tables with the same columns as `train`, `valid`, and `test`, where each feature column has been converted to standard units using the training set's mean and standard deviation. The `subject` and `group` columns should be unchanged.

_Points:_ 4

In [ ]:
...

# Keep this code to show the first 3 rows of the standardized train table.
train_std.show(3)

In [ ]:
grader.check("task_03")

---

## Building the Classifier

Recall the idea of $k$-nearest neighbors: to classify a new person, find the $k$ people in the training set whose features are closest to theirs, and predict the group that most of those neighbors belong to.

### Task 04 📍

Define a function `distance` that takes two arrays of the same length and returns the Euclidean (straight-line) distance between them.

_Points:_ 2

In [ ]:
def distance(arr1, arr2):
    ...

# Don't change the following code.
distance(make_array(1, 2), make_array(4, 6))

In [ ]:
grader.check("task_04")

---

### Task 05 📍

Define a function `classify` that takes four arguments:

* `training`: a table of people with known groups, such as `train_std`.
* `row`: the features of one new person, given as a row of the feature columns.
* `k`: the number of neighbors to use.
* `features`: an array of column labels that represent the features. (e.g., `feature_labels`)

The function should find the `k` rows of `training` whose features are closest to `row`, and return the most common `group` among those `k` neighbors.

**Hints**: 
1. Find the distance from `row` to every training person, add those distances as a column called `'Distance'`, sort the table, keep the first `k` rows, and then take a majority vote.
2. When you calculate the distances, use the feature columns of the `training` table.
3. You can convert a row named `a_row` to an array using `np.array(a_row)`.

_Points:_ 3

In [ ]:
def classify(training, row, k, features):
    ...

# Don't change the following code.
first_prediction = classify(train_std, test_std.select(feature_labels).row(0), 5, feature_labels)
first_prediction

In [ ]:
grader.check("task_05")

---

### Task 06 📍

A "lazy" classifier ignores the features completely and predicts the most common group in the training set for everyone. This is a useful baseline: a real classifier should beat it.

Assign `lazy_accuracy` to the proportion of people in `test` who would be classified correctly by this lazy classifier.

_Points:_ 2

In [ ]:
...
lazy_accuracy

In [ ]:
grader.check("task_06")

---

### Task 07 📍

Define a function `accuracy` with four arguments that takes a training table, a table to evaluate on, a number `k`, and the feature labels. It should classify every person in the evaluation table using the training table and return the **proportion** of those people classified correctly. It should be a proportion, not a percentage.

**Hint**: Use the `classify` function you wrote in Task 05.

_Points:_ 2

In [ ]:
...

In [ ]:
grader.check("task_07")

---

### Task 08 📍

Using the `valid_std` and $k = 7$:

1. Assign `two_feature_labels` to an array with just `'A'` and `'B'`.
2. Assign `two_feature_accuracy` to the accuracy using only those two features.
3. Assign `four_feature_accuracy` to the accuracy using all four features.

_Points:_ 2

In [ ]:
two_feature_labels = ...
two_feature_accuracy = ...
four_feature_accuracy = ...

# Do not change the following code.
print(f'Validation accuracy with 2 features: {two_feature_accuracy:.3f}.  Validation accuracy with 4 features: {four_feature_accuracy:.3f}.')

In [ ]:
grader.check("task_08")

---

## Choosing $k$ With the Validation Set

Now you need to choose $k$. It is tempting to try many values and keep the one that does best on the test set. But then the test set would influence your choice, and your final accuracy would be overly optimistic.

Instead, use the **validation set**. For each candidate value of $k$, classify every validation person using the training set and record the accuracy. Then choose the $k$ with the highest validation accuracy. Only after that should you use the test set.

### Task 09📍

Create a table called `k_results` with two columns: `'k'`, containing the odd numbers from 1 to 21, and `'Accuracy'`, containing the accuracy on the validation set using `train_std` as the training set, `valid_std` as the set to evaluate, and all four features, for each value of `k`. Use the `accuracy` function from Task 07. Then assign `best_k` to the value of `k` with the highest validation accuracy.

**Note**: This code should take only a few seconds to run.

_Points:_ 4

In [ ]:
k_values = np.arange(1, 22, 2)
...
best_k

In [ ]:
grader.check("task_09")

---

### Task 10 📍🔎

<!-- BEGIN QUESTION -->

Create a line plot of the validation accuracy (vertical axis) against `k` (horizontal axis) using `k_results`.

_Points:_ 1

In [ ]:
...

# Keep this code to give your image a title
plt.title('Validation Accuracy by k')
plt.show()

<!-- END QUESTION -->

---

## Final Evaluation

Now that you have chosen $k$ with the validation set, evaluate your classifier once on the test set. The classifier still uses only the training set to make predictions.

### Task 11 📍

Assign `final_accuracy` to the accuracy of your classifier on `test_std`, using `best_k` and all four features.

_Points:_ 2

In [ ]:
...
final_accuracy

In [ ]:
grader.check("task_11")

---

### How Precise Is Your Accuracy?

Your test set has only 24 people. If you tested the classifier on 24 _different_ people, you might get a different accuracy. You can use the bootstrap to estimate how much the accuracy might vary.

### Task 12 📍

* Assign `test_correct` to an array with one element for each person in `test_std`: `True` if the classifier (with `best_k` and all four features) classified that person correctly, and `False` otherwise.
* Assign `boot_accuracies` to an array of 2,000 bootstrap accuracies. Each one is the proportion of `True` values in a resample, with replacement, of `test_correct` of the same size.
* Assign `accuracy_ci_lower` and `accuracy_ci_upper` to the endpoints of an approximate 95% confidence interval for the classifier's accuracy.

**Hint**: `np.random.choice(arr, len(arr))` resamples an array with replacement.

_Points:_ 3

In [ ]:
...

# Leave this code to print out the 95% confidence interval.
print(f'Approximate 95% confidence interval for the accuracy: [{accuracy_ci_lower:.3f}, {accuracy_ci_upper:.3f}]')

In [ ]:
grader.check("task_12")

---

### Task 13 📍🔎

<!-- BEGIN QUESTION -->

The researchers reported about 89% accuracy, but they used a different method of classification. Your test accuracy is 75%, and the bootstrap interval from Task 12 gives a range of plausible accuracies. Test the null hypothesis H0: the true accuracy of your k-NN classifier is 0.89, against the alternative H1: it is not 0.89. State your conclusion in terms of your classifier's accuracy, and explain what the test does not show about the researchers' model.

_Points:_ 2

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---

## Final Task 📍

<!-- BEGIN QUESTION -->

You're almost done! Run the cell below to check that you passed all of the non-hidden auto-graded tasks, and then follow the instructions below to submit your assignment to Canvas.

_Points:_ 0

In [ ]:
grader.check_all()

<!-- END QUESTION -->

---

## Submit Your Assignment to Canvas

Follow these steps to submit your homework assignment:

1. **Review the Rubric:** View the rubric on the associated Canvas Assignment page to understand the scoring criteria.
2. **Run the Auto-Grader:** Ensure you have executed the code cell containing the command `grader.check_all()` to run all tests for auto-graded tasks marked with 📍. This command will execute all auto-grader tests sequentially.
3. **Complete Manually Graded Tasks:** Verify that you have responded to all the manually graded tasks marked with 📍🔎.
4. **Save Your Work:** In the notebook's Toolbar, go to `File -> Save default` to save your work and create a checkpoint.
5. **Download the Notebook:** In the notebook's Toolbar, go to `File -> Download IPYNB` to download the notebook (`.ipynb`) file.
6. **Upload to Canvas:** On the Canvas Assignment page, click "Start Assignment" or "New Attempt" to upload the downloaded `.ipynb` file.

---

## Attribution

This content is licensed under the <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0)</a> and derived from the <a href="https://www.data8.org/">Data 8: The Foundations of Data Science</a> offered by the University of California, Berkeley.

<img src="./by-nc-sa.png" width=100px>